In [1]:
import glob
import re
import os

import cartopy.crs as ccrs
import cmocean.cm as cmo
from IPython.display import HTML
import jax
import jax.numpy as jnp
from matplotlib import animation
import matplotlib.pyplot as plt
import numpy as np
import optax
import xarray as xr

from jaxparrow import fixed_point, gradient_wind, minimization_based
from jaxparrow.utils.kinematics import kinetic_energy, strain_rate, vorticity


jax.config.update("jax_enable_x64", True)

In [2]:
def get_pass_nums():
    pattern = os.path.join("./data", "SWOT_L3_Expert_v3_0_pass*.zarr")
    regex = re.compile(r"SWOT_L3_Expert_v3_0_pass(\d+)\.zarr")

    pass_nums = sorted({regex.match(os.path.basename(f)).group(1)
                        for f in glob.glob(pattern)
                        if regex.match(os.path.basename(f))})
    
    return pass_nums


def open_datasets():
    pass_nums = get_pass_nums()
    return [xr.open_zarr(f"./data/SWOT_L3_Expert_v3_0_pass{pass_num}.zarr") for pass_num in pass_nums]

In [3]:
datasets = open_datasets()

In [ ]:
def merge_passes(dataarrays):
    chunks = []
    offset = 0
    for da in dataarrays:
        n = da.sizes["num_lines"]
        da = (
            da
            .drop_vars("time", errors="ignore")
            .assign_coords(num_lines=np.arange(offset, offset + n))
        )
        chunks.append(da)
        offset += n
    return xr.concat(chunks, dim="num_lines")


def do_animation(datasets, varname, cmap, label, vmin=None, vmax=None, extend=None, figsize=(15, 5), interval=500):
    plt.ioff()

    from_time = min([min(ds.cycle_num).values for ds in datasets])
    to_time = max([max(ds.cycle_num) for ds in datasets]).values

    dataarrays = [ds[varname] for ds in datasets]
    da = merge_passes(dataarrays)

    da = da.isel(cycle_num=slice(0,1))

    if vmin is None and vmax is None:
        arr = da.values
        vmin, vmax = np.nanpercentile(arr, [2, 98])
        extend = "both"

    qm = da.sel(cycle_num=from_time).plot.pcolormesh(
        x="longitude", y="latitude",
        cmap=cmap, vmin=vmin, vmax=vmax, extend=extend, cbar_kwargs={"label": label},
        figsize=figsize, subplot_kws=dict(projection=ccrs.PlateCarree), transform=ccrs.PlateCarree(), 
    )

    fig = qm.figure
    ax = qm.axes

    ax.coastlines()
    title = ax.set_title(f"Cycle: {str(from_time)}")

    def draw_frame(i):
        try:
            qm.set_array(da.sel(cycle_num=i).values.ravel())
        except:
            qm.set_array(np.full_like(da.isel(cycle_num=0), np.nan).ravel())

        title.set_text(f"Cycle: {str(i)}")

        return qm, title
    
    frames = np.arange(from_time, to_time + 1)

    anim = animation.FuncAnimation(
        fig, draw_frame, init_func=lambda: draw_frame(from_time), frames=frames, interval=interval, blit=True
    )

    video = anim.to_html5_video()

    plt.close(fig)
    return HTML(video)

In [ ]:
plt.ioff()
do_animation(datasets, varname="ssh_filtered", cmap=cmo.deep_r, label="SSH (m)", interval=500)

In [ ]:
def create_template(ds):
    template_ds = xr.full_like(ds, fill_value=np.nan)
    template_arr = xr.full_like(template_ds.ssh_filtered, fill_value=np.nan)

    for name in [
        "ucg_mb", "vcg_mb", "ucg_gw", "vcg_gw", "ucg_fp", "vcg_fp",
        "ucga_mb", "vcga_mb", "ucga_gw", "vcga_gw", "ucga_fp", "vcga_fp",
        "nrv_g", "nrv_cg_mb", "nrv_cg_gw", "nrv_cg_fp",
        "ke_g", "ke_cg_mb", "ke_cg_gw", "ke_cg_fp",
        "eke_g", "eke_cg_mb", "eke_cg_gw", "eke_cg_fp",
    ]:
        template_ds[name] = template_arr

    return template_ds.chunk({"cycle_num": 1, "num_lines": -1, "num_pixels": -1})

In [ ]:
def arr_to_da(arr, long_name, comment, units=None):
    attrs = {"long_name": long_name, "comment": comment}
    if units is not None:
        attrs["units"] = units

    return xr.DataArray(
        arr[None, :, :], dims=("cycle_num", "num_lines", "num_pixels"), attrs=attrs
    )

In [ ]:
def estimate_cyclogeostrophy(ds):
    _ds = ds.isel(cycle_num=0)

    optim = optax.sgd(learning_rate=5e-3)
    optim = optax.chain(optax.clip(1), optim)

    lat = jnp.asarray(_ds.latitude)
    lon = jnp.asarray(_ds.longitude)
    ug = jnp.asarray(_ds.ugos_filtered.values)
    vg = jnp.asarray(_ds.vgos_filtered.values)
    uga = jnp.asarray(_ds.ugosa_filtered.values)
    vga = jnp.asarray(_ds.vgosa_filtered.values)

    mb_results = minimization_based(lat_t=lat, lon_t=lon, ug_t=ug, vg_t=vg, optim=optim)
    gw_results = gradient_wind(lat_t=lat, lon_t=lon, ug_t=ug, vg_t=vg)
    fp_results = fixed_point(lat_t=lat, lon_t=lon, ug_t=ug, vg_t=vg)

    ucg_mb = mb_results.ucg
    vcg_mb = mb_results.vcg
    ucg_gw = gw_results.ucg
    vcg_gw = gw_results.vcg
    ucg_fp = fp_results.ucg
    vcg_fp = fp_results.vcg

    mba_results = minimization_based(lat_t=lat, lon_t=lon, ug_t=uga, vg_t=vga, optim=optim)
    gwa_results = gradient_wind(lat_t=lat, lon_t=lon, ug_t=uga, vg_t=vga)
    fpa_results = fixed_point(lat_t=lat, lon_t=lon, ug_t=uga, vg_t=vga)

    ucga_mb = mba_results.ucg
    vcga_mb = mba_results.vcg
    ucga_gw = gwa_results.ucg
    vcga_gw = gwa_results.vcg
    ucga_fp = fpa_results.ucg
    vcga_fp = fpa_results.vcg

    nrv_g, nrv_cg_mb, nrv_cg_gw, nrv_cg_fp = map(
        lambda u, v: vorticity(u, v, lat, lon), (ug, ucg_mb, ucg_gw, ucg_fp), (vg, vcg_mb, vcg_gw, vcg_fp)
    )

    ke_g, ke_cg_mb, ke_cg_gw, ke_cg_fp = map(
        lambda u, v: kinetic_energy(u, v), (ug, ucg_mb, ucg_gw, ucg_fp), (vg, vcg_mb, vcg_gw, vcg_fp)
    )
    eke_g, eke_cg_mb, eke_cg_gw, eke_cg_fp = map(
        lambda u, v: kinetic_energy(u, v), (uga, ucga_mb, ucga_gw, ucga_fp), (vga, vcga_mb, vcga_gw, vcga_fp)
    )

    ds["ucg_mb"] = arr_to_da(
        ucg_mb, 
        "Cyclogeostrophic velocity: zonal component", 
        "Computed using the minimization-based method from ugos_filtered and vgos_filtered", 
        "m/s"
    )
    ds["vcg_mb"] = arr_to_da(
        vcg_mb,
        "Cyclogeostrophic velocity: meridional component",
        "Computed using the minimization-based method from ugos_filtered and vgos_filtered",
        "m/s"
    )
    ds["ucg_gw"] = arr_to_da(
        ucg_gw, 
        "Cyclogeostrophic velocity: meridional component",
        "Computed using the gradient-wind method from ugos_filtered and vgos_filtered",
        "m/s"
    )
    ds["vcg_gw"] = arr_to_da(
        vcg_gw, 
        "Cyclogeostrophic velocity: meridional component",
        "Computed using the gradient-wind method from ugos_filtered and vgos_filtered",
        "m/s"
    )
    ds["ucg_fp"] = arr_to_da(
        ucg_fp, 
        "Cyclogeostrophic velocity: zonal component",
        "Computed using the fixed-point method from ugos_filtered and vgos_filtered",
        "m/s"
    )
    ds["vcg_fp"] = arr_to_da(
        vcg_fp, 
        "Cyclogeostrophic velocity: meridional component",
        "Computed using the fixed-point method from ugos_filtered and vgos_filtered",
        "m/s"
    )

    ds["ucga_mb"] = arr_to_da(
        ucga_mb, 
        "Cyclogeostrophic velocity anomaly: zonal component", 
        "Computed using the minimization-based method from ugosa_filtered and vgosa_filtered", 
        "m/s"
    )
    ds["vcga_mb"] = arr_to_da(
        vcga_mb,
        "Cyclogeostrophic velocity anomaly: meridional component",
        "Computed using the minimization-based method from ugosa_filtered and vgosa_filtered",
        "m/s"
    )
    ds["ucga_gw"] = arr_to_da(
        ucga_gw, 
        "Cyclogeostrophic velocity anomaly: meridional component",
        "Computed using the gradient-wind method from ugosa_filtered and vgosa_filtered",
        "m/s"
    )
    ds["vcga_gw"] = arr_to_da(
        vcga_gw, 
        "Cyclogeostrophic velocity anomaly: meridional component",
        "Computed using the gradient-wind method from ugosa_filtered and vgosa_filtered",
        "m/s"
    )
    ds["ucga_fp"] = arr_to_da(
        ucga_fp, 
        "Cyclogeostrophic velocity anomaly: zonal component",
        "Computed using the fixed-point method from ugosa_filtered and vgosa_filtered",
        "m/s"
    )
    ds["vcga_fp"] = arr_to_da(
        vcga_fp, 
        "Cyclogeostrophic velocity anomaly: meridional component",
        "Computed using the fixed-point method from ugosa_filtered and vgosa_filtered",
        "m/s"
    )

    ds["nrv_g"] = arr_to_da(
        nrv_g, 
        "Normalized relative vorticity of the geostrophic flow",
        "Computed from ugos_filtered and vgos_filtered",
        ""
    )
    ds["nrv_cg_mb"] = arr_to_da(
        nrv_cg_mb, 
        "Normalized relative vorticity of the cyclogeostrophic flow",
        "Computed using the minimization-based method from ucg_mb and vcg_mb",
        ""
    )
    ds["nrv_cg_gw"] = arr_to_da(
        nrv_cg_gw, 
        "Normalized relative vorticity of the cyclogeostrophic flow",
        "Computed using the gradient-wind method from ucg_gw and vcg_gw",
        ""
    )
    ds["nrv_cg_fp"] = arr_to_da(
        nrv_cg_fp, 
        "Normalized relative vorticity of the cyclogeostrophic flow",
        "Computed using the fixed-point method from ucg_fp and vcg_fp",
        ""
    )

    ds["ke_g"] = arr_to_da(
        ke_g, 
        "Kinetic energy of the geostrophic flow",
        "Computed from ugos_filtered and vgos_filtered",
        "m$^2$ s$^{-2}$"
    )
    ds["ke_cg_mb"] = arr_to_da(
        ke_cg_mb, 
        "Kinetic energy of the cyclogeostrophic flow",
        "Computed using the minimization-based method from ucg_mb and vcg_mb",
        "m$^2$ s$^{-2}$"
    )
    ds["ke_cg_gw"] = arr_to_da(
        ke_cg_gw, 
        "Kinetic energy of the cyclogeostrophic flow",
        "Computed using the gradient-wind method from ucg_gw and vcg_gw",
        "m$^2$ s$^{-2}$"
    )
    ds["ke_cg_fp"] = arr_to_da(
        ke_cg_fp, 
        "Kinetic energy of the cyclogeostrophic flow",
        "Computed using the fixed-point method from ucg_fp and vcg_fp",
        "m$^2$ s$^{-2}$"
    )
    ds["eke_g"] = arr_to_da(
        eke_g, 
        "Eddy kinetic energy of the geostrophic flow",
        "Computed from ugosa_filtered and vgosa_filtered",
        "m$^2$ s$^{-2}$"
    )
    ds["eke_cg_mb"] = arr_to_da(
        eke_cg_mb, 
        "Eddy kinetic energy of the cyclogeostrophic flow",
        "Computed using the minimization-based method from ucga_mb and vcga_mb",
        "m$^2$ s$^{-2}$"
    )
    ds["eke_cg_gw"] = arr_to_da(
        eke_cg_gw, 
        "Eddy kinetic energy of the cyclogeostrophic flow",
        "Computed using the gradient-wind method from ucga_gw and vcga_gw",
        "m$^2$ s$^{-2}$"
    )
    ds["eke_cg_fp"] = arr_to_da(
        eke_cg_fp, 
        "Eddy kinetic energy of the cyclogeostrophic flow",
        "Computed using the fixed-point method from ucga_fp and vcga_fp",
        "m$^2$ s$^{-2}$"
    )

    return ds

In [ ]:
expert_003_ds, expert_016_ds, mrcosts_003_ds, mrcosts_016_ds = map(
    lambda ds: xr.map_blocks(
        estimate_cyclogeostrophy, 
        ds.chunk({"cycle_num": 1, "num_lines": -1, "num_pixels": -1}), 
        template=create_template(ds)
    ),
    (expert_003_ds, expert_016_ds, mrcosts_003_ds, mrcosts_016_ds),
)

## Strain rate and Okubo-Weiss parameter

In [ ]:
def create_strain_ow_template(ds):
    template_ds = xr.full_like(ds, fill_value=np.nan)
    template_arr = xr.full_like(template_ds.ssh_filtered, fill_value=np.nan)
    for name in [
        "strain_g", "strain_cg_mb", "strain_cg_gw", "strain_cg_fp",
        "ow_g", "ow_cg_mb", "ow_cg_gw", "ow_cg_fp",
    ]:
        template_ds[name] = template_arr
    return template_ds.chunk({"cycle_num": 1, "num_lines": -1, "num_pixels": -1})


def compute_strain_and_ow(ds):
    _ds = ds.isel(cycle_num=0)

    lat = jnp.asarray(_ds.latitude)
    lon = jnp.asarray(_ds.longitude)

    uv_pairs = [
        ("ugos_filtered", "vgos_filtered"),
        ("ucg_mb", "vcg_mb"),
        ("ucg_gw", "vcg_gw"),
        ("ucg_fp", "vcg_fp"),
    ]
    nrv_names = ["nrv_g", "nrv_cg_mb", "nrv_cg_gw", "nrv_cg_fp"]
    strain_names = ["strain_g", "strain_cg_mb", "strain_cg_gw", "strain_cg_fp"]
    ow_names = ["ow_g", "ow_cg_mb", "ow_cg_gw", "ow_cg_fp"]
    method_labels = ["geostrophic", "minimization-based", "gradient-wind", "fixed-point"]

    for (u_var, v_var), nrv_name, strain_name, ow_name, method in zip(
        uv_pairs, nrv_names, strain_names, ow_names, method_labels
    ):
        u = jnp.asarray(_ds[u_var].values)
        v = jnp.asarray(_ds[v_var].values)

        s = strain_rate(u, v, lat, lon)
        nrv = jnp.asarray(_ds[nrv_name].values)
        ow = s ** 2 - nrv ** 2

        ds[strain_name] = arr_to_da(
            s,
            f"Normalized strain rate of the {method} flow",
            f"Computed from {u_var} and {v_var}, normalized by Coriolis",
            "",
        )
        ds[ow_name] = arr_to_da(
            ow,
            f"Normalized Okubo-Weiss parameter of the {method} flow",
            f"OW/f² = (|S|/f)² - (ζ/f)², from {strain_name} and {nrv_name}",
            "",
        )

    return ds

In [ ]:
expert_003_ds, expert_016_ds, mrcosts_003_ds, mrcosts_016_ds = map(
    lambda ds: xr.map_blocks(
        compute_strain_and_ow,
        ds.chunk({"cycle_num": 1, "num_lines": -1, "num_pixels": -1}),
        template=create_strain_ow_template(ds),
    ),
    (expert_003_ds, expert_016_ds, mrcosts_003_ds, mrcosts_016_ds),
)

In [ ]:
expert_003_ds.chunk({"cycle_num": 10}).to_zarr("data/SWOT_L3_Expert_v3_0_pass003_cg.zarr", mode="w")
expert_016_ds.chunk({"cycle_num": 10}).to_zarr("data/SWOT_L3_Expert_v3_0_pass016_cg.zarr", mode="w")
mrcosts_003_ds.chunk({"cycle_num": 10}).to_zarr("data/SWOT_L3_mrCOSTS_pass003_cg.zarr", mode="w")
mrcosts_016_ds.chunk({"cycle_num": 10}).to_zarr("data/SWOT_L3_mrCOSTS_pass016_cg.zarr", mode="w")

/Users/bertrava/miniforge3/envs/swot-cyclogeo/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/Users/bertrava/miniforge3/envs/swot-cyclogeo/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/Users/bertrava/miniforge3/envs/swot-cyclogeo/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/Users/bertrava/miniforge3/envs/swot-cyclogeo/lib/python3.11/site-packages/zarr/api/asynchr